# NSR Mock Classification — Analysis\n\nAnalyses LLM-assigned New Social Risk (NSR) labels from the combined COVID + Great Recession mock run.\n\n**Input:** `data/processed/germany/2026-05-12_germany_mock_nsr_both_crises_llm_nsr_gpt-4.1-mini_v1_zero_shot.json`

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

DATA_FILE = (
    Path("..") / "data" / "processed" / "germany"
    / "2026-05-12_germany_mock_nsr_both_crises_llm_nsr_gpt-4.1-mini_v1_zero_shot.json"
)

raw = json.loads(DATA_FILE.read_text(encoding="utf-8"))
entries = raw["entries"]

df = pd.DataFrame([
    {
        "id":      e["id"],
        "crisis":  e.get("crisis", "unknown"),
        "year":    e.get("year"),
        "nsr":     e["nsr_labels"]["nsr"]["value"] if e.get("nsr_labels") else None,
    }
    for e in entries
])

# short labels for readability in plots
SHORT = {
    "reconciling work and family life":  "Work-family\nreconciliation",
    "single parenthood":                 "Single\nparenthood",
    "having a frail relative":           "Frail\nrelative",
    "possessing low or obsolete skills": "Low/obsolete\nskills",
    "insufficient social security coverage": "Insufficient\nSS coverage",
    "na":                                "n/a",
}
CATEGORY_ORDER = list(SHORT.keys())

df["nsr_short"] = df["nsr"].map(SHORT)

# palette — consistent across all plots
PALETTE = {
    "Work-family\nreconciliation": "#4C72B0",
    "Single\nparenthood":          "#DD8452",
    "Frail\nrelative":             "#55A868",
    "Low/obsolete\nskills":        "#C44E52",
    "Insufficient\nSS coverage":  "#8172B2",
    "n/a":                         "#CCCCCC",
}

print(f"Total entries: {len(df)}")
print(df["crisis"].value_counts().to_string())
print("\nOverall NSR distribution:")
print(df["nsr"].value_counts().to_string())

## 1 · Overall NSR category counts (all documents)

In [ ]:
short_order = [SHORT[c] for c in CATEGORY_ORDER]
counts = df["nsr_short"].value_counts().reindex(short_order, fill_value=0)
colors = [PALETTE[c] for c in short_order]

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(counts.index, counts.values, color=colors, edgecolor="white", linewidth=0.8)

ax.bar_label(bars, padding=3, fontsize=10)
ax.set_xlabel("NSR category", labelpad=8)
ax.set_ylabel("Number of documents")
ax.set_title("New Social Risk targeting — all documents (COVID + Great Recession)", pad=12)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.spines[["top", "right"]].set_visible(False)
ax.set_ylim(0, counts.max() * 1.12)

plt.tight_layout()
plt.show()

## 2 · NSR shares by crisis period

In [ ]:
CRISIS_LABELS = {
    "covid_2019-2022":             "COVID-19\n(2019–2022)",
    "great_recession_2008-2015":   "Great Recession\n(2008–2015)",
}

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, (crisis_key, crisis_label) in zip(axes, CRISIS_LABELS.items()):
    subset = df[df["crisis"] == crisis_key]["nsr_short"]
    counts_c = subset.value_counts().reindex(short_order, fill_value=0)

    # drop zero-count categories so the pie stays readable
    counts_c = counts_c[counts_c > 0]
    pie_colors = [PALETTE[c] for c in counts_c.index]

    wedges, texts, autotexts = ax.pie(
        counts_c.values,
        labels=None,
        colors=pie_colors,
        autopct=lambda p: f"{p:.1f}%\n(n={round(p * counts_c.sum() / 100):.0f})",
        pctdistance=0.72,
        startangle=90,
        wedgeprops={"edgecolor": "white", "linewidth": 1.2},
    )
    for at in autotexts:
        at.set_fontsize(8.5)

    ax.legend(
        wedges,
        counts_c.index,
        title="NSR category",
        loc="lower center",
        bbox_to_anchor=(0.5, -0.22),
        ncol=2,
        fontsize=8.5,
        title_fontsize=9,
        frameon=False,
    )
    ax.set_title(crisis_label, fontsize=13, pad=14)

fig.suptitle("New Social Risk targeting by crisis period", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3 · Policies targeting frail relatives, single parenthood, or low/obsolete skills

In [4]:
from IPython.display import display, HTML

FOCUS_NSR = {
    "having a frail relative":           "Frail relative",
    "single parenthood":                 "Single parenthood",
    "possessing low or obsolete skills": "Low / obsolete skills",
}

CRISIS_DISPLAY = {
    "covid_2019-2022":           "COVID-19 (2019–2022)",
    "great_recession_2008-2015": "Great Recession (2008–2015)",
}

df_detail = pd.DataFrame([
    {
        "crisis":       e.get("crisis", "unknown"),
        "year":         e.get("year"),
        "nsr":          e["nsr_labels"]["nsr"]["value"] if e.get("nsr_labels") else None,
        "title":        e.get("title", ""),
        "justification": (
            e["nsr_labels"]["nsr_justification"]["value"]
            if e.get("nsr_labels") else ""
        ),
    }
    for e in entries
])

focus = (
    df_detail[df_detail["nsr"].isin(FOCUS_NSR)]
    .assign(
        Crisis=lambda d: d["crisis"].map(CRISIS_DISPLAY),
        Category=lambda d: d["nsr"].map(FOCUS_NSR),
    )
    .sort_values(["crisis", "nsr", "year"])
    [["Crisis", "Category", "year", "title", "justification"]]
    .rename(columns={"year": "Year", "title": "Title", "justification": "Justification"})
    .reset_index(drop=True)
)

# ── HTML table with text-wrap and group shading ───────────────────────────────
CATEGORY_COLORS = {
    "Frail relative":        "#e8f4e8",
    "Single parenthood":     "#fef3e8",
    "Low / obsolete skills": "#fde8e8",
}

def row_style(row):
    bg = CATEGORY_COLORS.get(row["Category"], "white")
    return [f"background-color: {bg}"] * len(row)

styled = (
    focus.style
    .apply(row_style, axis=1)
    .set_properties(**{
        "text-align":   "left",
        "white-space":  "normal",
        "word-wrap":    "break-word",
        "max-width":    "480px",
        "font-size":    "13px",
        "padding":      "6px 10px",
    })
    .set_properties(subset=["Year"], **{"max-width": "50px", "text-align": "center"})
    .set_properties(subset=["Category"], **{"font-weight": "600", "max-width": "140px"})
    .set_table_styles([
        {"selector": "thead th", "props": [
            ("background-color", "#2c3e50"), ("color", "white"),
            ("font-size", "13px"), ("padding", "8px 10px"), ("text-align", "left"),
        ]},
        {"selector": "tbody tr:hover", "props": [("filter", "brightness(0.96)")]},
        {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%")]},
        {"selector": "td, th", "props": [("border", "1px solid #ddd")]},
    ])
    .hide(axis="index")
    .set_caption(
        f"<b>Policies targeting frail relatives, single parenthood, or low/obsolete skills</b>"
        f"  —  n = {len(focus)}"
    )
)

display(styled)

Crisis,Category,Year,Title,Justification
COVID-19 (2019–2022),Frail relative,2020,Gesetz zur Erhöhung der Behinderten-Pauschbeträge und zur Anpassung weiterer steuerlicher Regelungen,"The law increases tax allowances and benefits for people with disabilities and for caregivers providing personal care, targeting those who care for frail or disabled relatives."
COVID-19 (2019–2022),Frail relative,2021,Gesetz zur Stärkung der Teilhabe von Menschen mit Behinderungen sowie zur landesrechtlichen Bestimmung der Träger von Leistungen für Bildung und Teilhabe in der Sozialhilfe (Teilhabestärkungsgesetz),"The legislation extensively addresses support and participation of people with disabilities, including digital care applications and assistance for care-dependent individuals, targeting those providing informal care or living with frail or disabled persons."
COVID-19 (2019–2022),Frail relative,2021,Verordnung zur Weiterführung der Ergänzenden unabhängigen Teilhabeberatung (Teilhabeberatungsverordnung – EUTBV),"The policy targets people with disabilities and those threatened by disability, providing independent participation counseling to support their rights and inclusion, which relates to having a frail relative."
COVID-19 (2019–2022),Frail relative,2022,Gesetz zur Verlängerung von Sonderregelungen im Zusammenhang mit der COVID-19-Pandemie beim Kurzarbeitergeld und anderer Leistungen,"The law extends special regulations related to digital care applications and support for care-dependent persons and their informal caregivers, targeting those providing unpaid care to frail relatives."
COVID-19 (2019–2022),Frail relative,2022,Gesetz zur Zahlung eines Bonus für Pflegekräfte in Krankenhäusern und Pflegeeinrichtungen (Pflegebonusgesetz),"The legislation primarily targets care workers and the provision of bonuses and support related to care during the COVID-19 pandemic, addressing the challenges faced by individuals providing care to frail or vulnerable persons."
COVID-19 (2019–2022),Low / obsolete skills,2020,Dritte Verordnung zur Anpassung der Höhe des Mindestlohns (Dritte Mindestlohnanpassungsverordnung – MiLoV3),"The policy adjusts the minimum wage, which primarily targets workers in low-wage sectors often characterized by low or obsolete skills."
COVID-19 (2019–2022),Low / obsolete skills,2022,Verordnung über die Berufsausbildung zur Fachkraft Küche (Fachkraft-Küche-Ausbildungsverordnung – FKüAusbV),"The regulation targets vocational training for kitchen specialists, aiming to equip individuals with specific skills and knowledge to improve employability in a low value added service sector."
Great Recession (2008–2015),Frail relative,2008,Verordnung zur Änderung arbeitszeit- und urlaubsrechtlicher Vorschriften,"The policy includes provisions for leave related to the care of sick or disabled children and requires medical certification for the necessity of care, targeting individuals providing informal care to frail relatives."
Great Recession (2008–2015),Frail relative,2008,Gesetz zur strukturellen Weiterentwicklung der Pflegeversicherung (Pflege-Weiterentwicklungsgesetz),"The Gesetz zur strukturellen Weiterentwicklung der Pflegeversicherung (Pflege-Weiterentwicklungsgesetz) primarily targets individuals providing care to frail elderly or disabled persons, focusing on the development and improvement of long-term care insurance and support structures for those with significant care needs."
Great Recession (2008–2015),Frail relative,2012,Gesetz zur Neuausrichtung der Pflegeversicherung (Pflege-Neuausrichtungs-Gesetz  PNG),"The Gesetz zur Neuausrichtung der Pflegeversicherung (Pflege-Neuausrichtungs-Gesetz – PNG) primarily targets individuals providing unpaid, informal care to frail elderly or disabled persons by restructuring long-term care insurance and enhancing home care and support services."
